# Module 04.17 — Cross-Type Restore Ordering

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.17 Cross-Type Restore Ordering

Kibana saved objects form a **directed acyclic dependency graph**. An object with entries
in its `references` array depends on all referenced objects existing before it can render
correctly. When Kibana performs a feature-state restore, it walks this graph and resolves
objects in dependency order — you never need to restore types one at a time.

```
Restore order (simplified):

  1. config, space, tag          (no dependencies)
  2. index-pattern               (no dependencies)
  3. query                       (no dependencies)
  4. search                      (depends on: index-pattern)
  5. visualization               (depends on: index-pattern, search)
  6. lens                        (depends on: index-pattern)
  7. map                         (depends on: index-pattern)
  8. canvas-workpad              (no external dependencies — self-contained)
  9. event-annotation-group      (depends on: index-pattern)
 10. action (connector)          (no dependencies)
 11. alert (rule)                (depends on: action)
 12. cases                       (depends on: action for connectors)
 13. cases-comments              (depends on: cases)
 14. cases-user-actions          (depends on: cases)
 15. dashboard                   (depends on: visualization, lens, map, search, tag)
 16. short-url                   (depends on: dashboard)
```

This ordering also explains why **partial restores are risky**. If you restore only a
dashboard without its referenced visualizations, the dashboard object will be written to
Kibana but its panels will render as errors. Kibana does not validate references at restore
time — it trusts you to restore a consistent set of objects.

The cells below demonstrate this with a full-state snapshot, a complete wipe of all Kibana
state, and a restore — then verify that object counts match before and after.

In [ ]:
heading('4.17 Full Kibana state snapshot → delete all → restore all')

# Take a comprehensive snapshot capturing everything we've built
snap_name = 'snap-full-kibana-state'
delete_snapshot_if_exists(es, SNAP_REPO, snap_name)
result = snapshot_kibana_state(es, SNAP_REPO, snap_name)
success(f'Full state snapshot: {result["state"]}')

# Count objects before delete
types_to_check = ['index-pattern', 'dashboard', 'lens', 'visualization', 'search',
                  'alert', 'action', 'tag', 'query', 'cases']
before_counts = {}
for t in types_to_check:
    before_counts[t] = len(find_saved_objects(t))

t_table = Table('Type', 'Before', 'After restore')
for type_name, count in before_counts.items():
    t_table.add_row(type_name, str(count), '...')
console.print(t_table)

In [ ]:
# Delete all Kibana sample data (this removes all saved objects from sample datasets)
for dataset in ['ecommerce', 'flights', 'logs']:
    try:
        remove_sample_data(dataset)
    except Exception as e:
        warn(f'Could not remove {dataset}: {e}')

# Also delete our training objects
for obj_type, obj_id in [
    ('tag', tag_id),
    ('alert', rule_id),
    ('action', connector_id),
]:
    try:
        kibana_delete(f'/api/saved_objects/{obj_type}/{obj_id}')
    except Exception:
        pass

time.sleep(2)

after_delete = {}
for t in types_to_check:
    after_delete[t] = len(find_saved_objects(t))

t2 = Table('Type', 'Before delete', 'After delete')
for type_name in types_to_check:
    t2.add_row(type_name, str(before_counts[type_name]), str(after_delete[type_name]))
console.print(t2)

In [ ]:
# Now restore from snapshot
heading('Restoring full Kibana state')
restore_kibana_state(es, SNAP_REPO, snap_name)
time.sleep(6)  # Give Kibana time to process the restored .kibana* indices

# Count after restore
after_restore = {}
for t in types_to_check:
    after_restore[t] = len(find_saved_objects(t))

t3 = Table('Type', 'Before', 'After delete', 'After restore', 'Match?')
for type_name in types_to_check:
    before = before_counts[type_name]
    after = after_restore[type_name]
    match = '[green]✓[/green]' if before == after else '[red]✗[/red]'
    t3.add_row(type_name, str(before), str(after_delete[type_name]), str(after), match)
console.print(t3)

## Summary

| Object type | Has external references? | Secrets encrypted? |
|------------|--------------------------|-------------------|
| `index-pattern` | No | No |
| `search` | Yes → `index-pattern` | No |
| `visualization` | Yes → `index-pattern`, `search` | No |
| `lens` | Yes → `index-pattern` | No |
| `map` | Yes → `index-pattern` | No |
| `dashboard` | Yes → visualizations, lens, maps, searches, tags | No |
| `canvas-workpad` | No (self-contained) | No |
| `tag` | No | No |
| `query` | No | No |
| `space` | No | No |
| `alert` (rule) | Yes → `action` | No |
| `action` (connector) | No | **Yes** — re-enter after cross-cluster restore |
| `cases` | Yes → `action` | No |
| `config` | Yes → `index-pattern` (defaultIndex) | No |
| `short-url` | Yes → target object | No |
| `event-annotation-group` | Yes → `index-pattern` | No |

**Next:** [05_restoring_snapshots.ipynb](05_restoring_snapshots.ipynb)